# Perceptron Multi-Couches (PMC) — Cas d'images


## 1. Imports


In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

# Chemin dynamique vers python
PYTHON_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'python'))
sys.path.insert(0, PYTHON_DIR)

# Import direct depuis pmc.py, comme pmc_cas_tests.py
from pmc import init, entrainer, predire, SCRIPT_DIR
from functions import load_dataset

DATASET_ROOT = os.path.join(SCRIPT_DIR, '..', 'dataset')
RESULTS_DIR  = os.path.join(SCRIPT_DIR, '..', 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

print('Imports OK')


Imports OK


In [6]:
# Chargement
train_folder = os.path.join(DATASET_ROOT, 'train_dataset')
test_folder  = os.path.join(DATASET_ROOT, 'test_dataset')

TAILLE     = 32
NB_CLASSES = 3
NOMS       = ['chat', 'chien', 'autre']

# 32x32
X_train_32_1, Y_train_32_1 = load_dataset(train_folder, target_size=(32,32), one_hot=True)
X_test_32_1,  Y_test_32_1  = load_dataset(test_folder,  target_size=(32,32), one_hot=True)
X_train_32_3, Y_train_32_3 = load_dataset(train_folder, target_size=(32,32), color=True, one_hot=True)
X_test_32_3,  Y_test_32_3  = load_dataset(test_folder,  target_size=(32,32), color=True, one_hot=True)

# 16x16
X_train_16_1, Y_train_16_1 = load_dataset(train_folder, target_size=(16,16), one_hot=True)
X_test_16_1,  Y_test_16_1  = load_dataset(test_folder,  target_size=(16,16), one_hot=True)
X_train_16_3, Y_train_16_3 = load_dataset(train_folder, target_size=(16,16), color=True, one_hot=True)
X_test_16_3,  Y_test_16_3  = load_dataset(test_folder,  target_size=(16,16), color=True, one_hot=True)

print(f'32x32 -- train: {len(X_train_32_1)} (gris) / {len(X_train_32_3)} (couleur)')
print(f'16x16 -- train: {len(X_train_16_1)} (gris) / {len(X_train_16_3)} (couleur)')


32x32 -- train: 2400 (gris) / 2400 (couleur)
16x16 -- train: 2400 (gris) / 2400 (couleur)


In [7]:
results = []

def run_pmc(X_train, Y_train, X_test, Y_test, resolution, couleur, epochs, alpha, nb_cachees=16, n_train=None, seed=42):
    # Mélange
    perm   = np.random.RandomState(seed).permutation(len(X_train))
    X_tr   = [X_train[i] for i in perm][:n_train]
    Y_tr   = [Y_train[i] for i in perm][:n_train]

    nb_entrees = resolution * resolution * (3 if couleur else 1)

    # Entrainement
    init(nb_entrees, nb_cachees, NB_CLASSES)
    entrainer(X_tr, Y_tr, epochs=epochs, alpha=alpha)

    # Precision
    def accuracy(X, Y):
        ok = sum(int(np.argmax(predire(x, NB_CLASSES))) == int(np.argmax(y)) for x, y in zip(X, Y))
        return 100 * ok / len(X)

    acc_tr = accuracy(X_tr, Y_tr)
    acc_te = accuracy(X_test, Y_test)

    canal = 'couleur' if couleur else 'gris'
    r = {'resolution': resolution, 'couleur': couleur, 'epochs': epochs,
         'alpha': alpha, 'nb_cachees': nb_cachees, 'n_train': len(X_tr),
         'train_acc': acc_tr, 'test_acc': acc_te}
    results.append(r)
    print(f'{resolution}x{resolution} {canal} | epochs={epochs} alpha={alpha} '
          f'n_train={len(X_tr)} -> train={acc_tr:.1f}% test={acc_te:.1f}%')

PEU_IMAGES  = 500
PEU_EPOCHS  = 200
PLUS_EPOCHS = 800
ALPHA       = 0.01


### 16x16 gris


In [8]:
run_pmc(X_train_16_1, Y_train_16_1, X_test_16_1, Y_test_16_1, 16, False, PEU_EPOCHS,  ALPHA, n_train=PEU_IMAGES)
run_pmc(X_train_16_1, Y_train_16_1, X_test_16_1, Y_test_16_1, 16, False, PEU_EPOCHS,  ALPHA)
run_pmc(X_train_16_1, Y_train_16_1, X_test_16_1, Y_test_16_1, 16, False, PLUS_EPOCHS, ALPHA)


16x16 gris | epochs=200 alpha=0.01 n_train=500 -> train=98.6% test=36.7%
16x16 gris | epochs=200 alpha=0.01 n_train=2400 -> train=85.1% test=39.8%
16x16 gris | epochs=800 alpha=0.01 n_train=2400 -> train=92.2% test=39.2%


### 16x16 couleur


In [9]:
run_pmc(X_train_16_3, Y_train_16_3, X_test_16_3, Y_test_16_3, 16, True, PEU_EPOCHS,  ALPHA, n_train=PEU_IMAGES)
run_pmc(X_train_16_3, Y_train_16_3, X_test_16_3, Y_test_16_3, 16, True, PEU_EPOCHS,  ALPHA)
run_pmc(X_train_16_3, Y_train_16_3, X_test_16_3, Y_test_16_3, 16, True, PLUS_EPOCHS, ALPHA)


16x16 couleur | epochs=200 alpha=0.01 n_train=500 -> train=97.4% test=40.5%
16x16 couleur | epochs=200 alpha=0.01 n_train=2400 -> train=89.4% test=47.2%
16x16 couleur | epochs=800 alpha=0.01 n_train=2400 -> train=93.6% test=44.8%


### 32x32 gris


In [10]:
run_pmc(X_train_32_1, Y_train_32_1, X_test_32_1, Y_test_32_1, 32, False, PEU_EPOCHS,  ALPHA, n_train=PEU_IMAGES)
run_pmc(X_train_32_1, Y_train_32_1, X_test_32_1, Y_test_32_1, 32, False, PEU_EPOCHS,  ALPHA)
run_pmc(X_train_32_1, Y_train_32_1, X_test_32_1, Y_test_32_1, 32, False, PLUS_EPOCHS, ALPHA)


32x32 gris | epochs=200 alpha=0.01 n_train=500 -> train=95.6% test=35.0%
32x32 gris | epochs=200 alpha=0.01 n_train=2400 -> train=92.6% test=37.7%
32x32 gris | epochs=800 alpha=0.01 n_train=2400 -> train=95.3% test=37.3%


### 32x32 couleur


In [11]:
run_pmc(X_train_32_3, Y_train_32_3, X_test_32_3, Y_test_32_3, 32, True, PEU_EPOCHS,  ALPHA, n_train=PEU_IMAGES)
run_pmc(X_train_32_3, Y_train_32_3, X_test_32_3, Y_test_32_3, 32, True, PEU_EPOCHS,  ALPHA)
run_pmc(X_train_32_3, Y_train_32_3, X_test_32_3, Y_test_32_3, 32, True, PLUS_EPOCHS, ALPHA)


32x32 couleur | epochs=200 alpha=0.01 n_train=500 -> train=93.0% test=37.8%
32x32 couleur | epochs=200 alpha=0.01 n_train=2400 -> train=90.2% test=45.0%
32x32 couleur | epochs=800 alpha=0.01 n_train=2400 -> train=94.8% test=42.5%


### Effet du learning rate


In [12]:
ALPHA_VALUES = [0.1, 0.01, 0.001, 0.0001, 0.00001, 0.00005]

print('===== 16x16 couleur | toutes les images | 200 epochs =====')
for a in ALPHA_VALUES:
    run_pmc(X_train_16_3, Y_train_16_3, X_test_16_3, Y_test_16_3, 16, True, PEU_EPOCHS, a)

print()
print('===== 32x32 couleur | toutes les images | 200 epochs =====')
for a in ALPHA_VALUES:
    run_pmc(X_train_32_3, Y_train_32_3, X_test_32_3, Y_test_32_3, 32, True, PEU_EPOCHS, a)


===== 16x16 couleur | toutes les images | 200 epochs =====
16x16 couleur | epochs=200 alpha=0.1 n_train=2400 -> train=55.8% test=49.2%
16x16 couleur | epochs=200 alpha=0.01 n_train=2400 -> train=90.0% test=47.8%
16x16 couleur | epochs=200 alpha=0.001 n_train=2400 -> train=71.0% test=49.5%
16x16 couleur | epochs=200 alpha=0.0001 n_train=2400 -> train=48.8% test=46.2%
16x16 couleur | epochs=200 alpha=1e-05 n_train=2400 -> train=38.5% test=35.7%
16x16 couleur | epochs=200 alpha=5e-05 n_train=2400 -> train=47.4% test=44.5%

===== 32x32 couleur | toutes les images | 200 epochs =====
32x32 couleur | epochs=200 alpha=0.1 n_train=2400 -> train=52.3% test=48.7%
32x32 couleur | epochs=200 alpha=0.01 n_train=2400 -> train=90.8% test=44.8%
32x32 couleur | epochs=200 alpha=0.001 n_train=2400 -> train=71.7% test=45.3%
32x32 couleur | epochs=200 alpha=0.0001 n_train=2400 -> train=52.3% test=48.3%
32x32 couleur | epochs=200 alpha=1e-05 n_train=2400 -> train=38.4% test=41.0%
32x32 couleur | epochs=200 

## 5. Récap final


In [13]:
top5 = sorted(results, key=lambda r: r['test_acc'], reverse=True)[:5]

print(f"{'Config':<16}{'N_train':>9}{'Epochs':>8}{'Alpha':>8}{'Train%':>9}{'Test%':>8}")
print('-' * 60)
for r in top5:
    canal  = 'couleur' if r['couleur'] else 'gris'
    config = f"{r['resolution']}x{r['resolution']} {canal}"
    print(f"{config:<16}{r['n_train']:>9}{r['epochs']:>8}{r['alpha']:>8}"
          f"{r['train_acc']:>8.1f}%{r['test_acc']:>7.1f}%")


Config            N_train  Epochs   Alpha   Train%   Test%
------------------------------------------------------------
16x16 couleur        2400     200   0.001    71.0%   49.5%
16x16 couleur        2400     200     0.1    55.8%   49.2%
32x32 couleur        2400     200     0.1    52.3%   48.7%
32x32 couleur        2400     200  0.0001    52.3%   48.3%
16x16 couleur        2400     200    0.01    90.0%   47.8%


## 6. Sauvegarde du meilleur modèle

On réentraîne avec la meilleure configuration et on sauvegarde via `py_sauvegarder`.


In [14]:
best  = top5[0]
canal = 'couleur' if best['couleur'] else 'gris'
print(f"Meilleure config : {best['resolution']}x{best['resolution']} {canal} | "
      f"epochs={best['epochs']} alpha={best['alpha']} -> test={best['test_acc']:.1f}%")


Meilleure config : 16x16 couleur | epochs=200 alpha=0.001 -> test=49.5%


In [ ]:
# Sélection du bon dataset
if best['resolution'] == 32 and best['couleur']:
    X_best, Y_best = X_train_32_3, Y_train_32_3
elif best['resolution'] == 32 and not best['couleur']:
    X_best, Y_best = X_train_32_1, Y_train_32_1
elif best['resolution'] == 16 and best['couleur']:
    X_best, Y_best = X_train_16_3, Y_train_16_3
else:
    X_best, Y_best = X_train_16_1, Y_train_16_1

# Réentraînement final
nb_entrees = best['resolution']**2 * (3 if best['couleur'] else 1)
init(nb_entrees, best['nb_cachees'], NB_CLASSES)
entrainer(X_best, Y_best, epochs=best['epochs'], alpha=best['alpha'])

# Sauvegarde
from pmc import lib as pmc_lib
chemin = os.path.join(SCRIPT_DIR, '..', 'models', 'pmc_chat_chien_autre.txt')
os.makedirs(os.path.dirname(chemin), exist_ok=True)
pmc_lib.py_sauvegarder(chemin.encode())
print(f'Modèle sauvegardé')


Modèle sauvegardé : c:\Users\Shun\Desktop\Projet_Annual_IA\python\..\models\pmc_chat_chien_autre.txt
